# Generate & Verify Model Weights

A small utility for producing a starting-weights file that pairs with a tracebloc model file.

Use this when:
- You wrote a custom model and need a `<modelname>_weights.pkl` to upload alongside it
- You want to confirm a weights file loads correctly before running `user.upload_model(..., weights=True)`

**Convention:** tracebloc expects a model file `mymodel.py` defining a function `MyModel()`, and a companion weights file named `mymodel_weights.pkl` (or `.pth` for PyTorch) in the same directory.

📖 [Model structure requirements](https://docs.tracebloc.io/join-use-case/model-optimization)

## 1. Point at your model file

Set the path to the `.py` file that defines `MyModel()`. The weights file will be written next to it.

In [ ]:
import importlib.util
import os

MODEL_PATH = "mymodel.py"  # <-- change to your model file

model_dir = os.path.dirname(os.path.abspath(MODEL_PATH)) or "."
model_name = os.path.splitext(os.path.basename(MODEL_PATH))[0]

spec = importlib.util.spec_from_file_location(model_name, MODEL_PATH)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

## 2. Generate weights

Pick the cell that matches your framework. Each one builds the model via `MyModel()` and writes a weights file using the format tracebloc expects.

### PyTorch

Saves the model's `state_dict` with `torch.save` — tracebloc loads it back via `model.load_state_dict(torch.load(...))`.

In [ ]:
import torch

weights_path = os.path.join(model_dir, f"{model_name}_weights.pth")

model = module.MyModel()
torch.save(model.state_dict(), weights_path)
print(f"Wrote {weights_path}")

### TensorFlow / Keras

Saves the weights as a pickled list of arrays — tracebloc loads it back via `model.set_weights(pickle.load(...))`.

In [ ]:
import pickle

weights_path = os.path.join(model_dir, f"{model_name}_weights.pkl")

model = module.MyModel()
with open(weights_path, "wb") as f:
    pickle.dump(model.get_weights(), f)
print(f"Wrote {weights_path}")

## 3. Verify the weights file

Reload the weights into a fresh model instance to confirm shapes match and the file is readable. Run the cell that matches your framework.

### PyTorch

In [ ]:
import torch

weights_path = os.path.join(model_dir, f"{model_name}_weights.pth")

verify_model = module.MyModel()
verify_model.load_state_dict(torch.load(weights_path))
print(f"Loaded weights into {type(verify_model).__name__}")
print(f"  parameters: {sum(p.numel() for p in verify_model.parameters()):,}")

### TensorFlow / Keras

In [ ]:
import pickle

weights_path = os.path.join(model_dir, f"{model_name}_weights.pkl")

verify_model = module.MyModel()
with open(weights_path, "rb") as f:
    verify_model.set_weights(pickle.load(f))
verify_model.summary()

## 4. Upload with your model

Once the weights file is in place, upload both together from the main training notebook:

```python
user.upload_model(MODEL_PATH, weights=True)
```

The SDK will look for `<modelname>_weights.pkl` (or `.pth`) next to your `.py` file.